In [ ]:

import pandas as pd
import numpy as np

print("Duka Electronics H1 Sales Exploration")
print("Period: January to June 2026")
print("Libraries loaded successfully.")


Duka Electronics H1 Sales Exploration
Period: January to June 2026
Libraries loaded successfully.


# Duka Electronics H1 Sales Exploration

**Period:** January–June 2026

**Objective:** To analyse sales performance across branches, products, sales channels, weekdays, weekends, and months to support management decision-making.

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Create folders required by the assignment
Path("outputs").mkdir(exist_ok=True)
Path("excel").mkdir(exist_ok=True)

print("Project folders created successfully.")
print("Ready to load the Duka Electronics dataset.")


Project folders created successfully.
Ready to load the Duka Electronics dataset.


In [ ]:
from google.colab import files

print("Duka Electronics assignment")
print("Next, we need the official PLP data generator.")
print("Required filename: generate_data.py")
print("Required random seed: 2026")

Duka Electronics assignment
Next, we need the official PLP data generator.
Required filename: generate_data.py
Required random seed: 2026


In [5]:
import os

print(os.listdir("/content"))

['.config', 'excel', 'outputs', 'sample_data']


In [13]:
import os

generator_path = "/content/sample_data/generate_data.py"

print("Generator exists:", os.path.exists(generator_path))

Generator exists: True


In [14]:
%run /content/sample_data/generate_data.py

Wrote duka_sales.csv with 1000 rows


In [15]:
import os

for root, folders, files in os.walk("/content"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/duka_sales.csv
/content/excel/premium_online_coast_capital.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv


In [16]:
import pandas as pd

df = pd.read_csv("/content/duka_sales.csv")

print("Dataset loaded successfully!")
print("Number of rows and columns:", df.shape)

display(df.head())

Dataset loaded successfully!
Number of rows and columns: (1000, 7)


,order_id,order_date,branch,product,channel,units,unit_price
0,DK0118,2026-01-01,Mombasa,Headphones,In-Store,4,6500
1,DK0287,2026-01-01,Eldoret,Power Bank,Online,3,2500
2,DK0079,2026-01-01,Nairobi,Smart TV,In-Store,2,45000
3,DK0938,2026-01-01,Mombasa,Laptop,In-Store,1,62000
4,DK0554,2026-01-02,Nakuru,Smartphone,In-Store,1,18500


In [17]:
# Check data types
print("DATA TYPES")
print(df.dtypes)

# Check missing values
print("\nMISSING VALUES")
print(df.isnull().sum())

# Check date range
df["order_date"] = pd.to_datetime(df["order_date"])
print("\nDATE RANGE")
print("First date:", df["order_date"].min())
print("Last date:", df["order_date"].max())

# Check categories
print("\nBRANCHES")
print(df["branch"].unique())

print("\nPRODUCTS")
print(df["product"].unique())

print("\nCHANNELS")
print(df["channel"].unique())

DATA TYPES
order_id      object
order_date    object
branch        object
product       object
channel       object
units          int64
unit_price     int64
dtype: object

MISSING VALUES
order_id      0
order_date    0
branch        0
product       0
channel       0
units         0
unit_price    0
dtype: int64

DATE RANGE
First date: 2026-01-01 00:00:00
Last date: 2026-06-30 00:00:00

BRANCHES
['Mombasa' 'Eldoret' 'Nairobi' 'Nakuru' 'Kisumu']

PRODUCTS
['Headphones' 'Power Bank' 'Smart TV' 'Laptop' 'Smartphone' 'Tablet']

CHANNELS
['In-Store' 'Online']


In [18]:

# Create five new columns

# 1. Total revenue for each order
df["revenue"] = df["units"] * df["unit_price"]

# 2. Month of the order
df["month"] = df["order_date"].dt.month_name()

# 3. Day of the week
df["weekday"] = df["order_date"].dt.day_name()

# 4. Identify weekend orders
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5

# 5. Group products by unit price
df["price_band"] = pd.cut(
    df["unit_price"],
    bins=[0, 5000, 20000, float("inf")],
    labels=["Budget", "Mid-range", "Premium"]
)

# Display the original and new columns
display(df.head(10))

print("Dataset shape:", df.shape)
print("\nNew columns created successfully!")
print(["revenue", "month", "weekday", "is_weekend", "price_band"])


,order_id,order_date,branch,product,channel,units,unit_price,revenue,month,weekday,is_weekend,price_band
0,DK0118,2026-01-01,Mombasa,Headphones,In-Store,4,6500,26000,January,Thursday,False,Mid-range
1,DK0287,2026-01-01,Eldoret,Power Bank,Online,3,2500,7500,January,Thursday,False,Budget
2,DK0079,2026-01-01,Nairobi,Smart TV,In-Store,2,45000,90000,January,Thursday,False,Premium
3,DK0938,2026-01-01,Mombasa,Laptop,In-Store,1,62000,62000,January,Thursday,False,Premium
4,DK0554,2026-01-02,Nakuru,Smartphone,In-Store,1,18500,18500,January,Friday,False,Mid-range
5,DK0606,2026-01-02,Mombasa,Headphones,Online,5,6500,32500,January,Friday,False,Mid-range
6,DK0979,2026-01-02,Nakuru,Power Bank,In-Store,2,2500,5000,January,Friday,False,Budget
7,DK0875,2026-01-02,Eldoret,Smartphone,Online,2,18500,37000,January,Friday,False,Mid-range
8,DK0483,2026-01-02,Nairobi,Smartphone,Online,1,18500,18500,January,Friday,False,Mid-range
9,DK0385,2026-01-02,Kisumu,Tablet,In-Store,1,28000,28000,January,Friday,False,Premium


Dataset shape: (1000, 12)

New columns created successfully!
['revenue', 'month', 'weekday', 'is_weekend', 'price_band']


In [20]:

# Total revenue and number of orders for each branch
branch_summary = df.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count")
).reset_index()

# Rank branches by revenue and orders
branch_summary["revenue_rank"] = (
    branch_summary["total_revenue"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

branch_summary["orders_rank"] = (
    branch_summary["total_orders"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Display the results
branch_summary = branch_summary.sort_values("revenue_rank")

display(branch_summary)


,branch,total_revenue,total_orders,revenue_rank,orders_rank
3,Nairobi,12133000,349,1,1
2,Mombasa,8444500,263,2,2
4,Nakuru,4392000,128,3,4
0,Eldoret,3860000,115,4,5
1,Kisumu,3805500,145,5,3


### Task 3: Branch Revenue and Order Rankings

- **Nairobi** ranked first in both revenue and orders, generating KSh 12,133,000 from 349 orders.
- **Mombasa** ranked second in both revenue and orders, generating KSh 8,444,500 from 263 orders.
- **Nakuru** ranked third in revenue with KSh 4,392,000, but ranked fourth in order count with 128 orders.
- **Kisumu** ranked fifth in revenue with KSh 3,805,500, but ranked third in order count with 145 orders.
- **Eldoret** ranked fourth in revenue with KSh 3,860,000 and fifth in order count with 115 orders.

**Explanation:** The rankings differ because branches do not generate the same revenue per order. Kisumu had more orders than Nakuru and Eldoret, but its total revenue was lower. This suggests that the average value of orders and the products purchased affect total branch revenue.

In [21]:

# Top 3 products by total revenue
product_revenue = (
    df.groupby("product")
    .agg(
        total_revenue=("revenue", "sum"),
        units_sold=("units", "sum")
    )
    .reset_index()
)

print("TOP 3 PRODUCTS BY REVENUE")
display(
    product_revenue
    .sort_values("total_revenue", ascending=False)
    .head(3)
)

print("TOP 3 PRODUCTS BY UNITS SOLD")
display(
    product_revenue
    .sort_values("units_sold", ascending=False)
    .head(3)
)


TOP 3 PRODUCTS BY REVENUE


,product,total_revenue,units_sold
1,Laptop,9300000,150
4,Smartphone,7159500,387
3,Smart TV,5355000,119


TOP 3 PRODUCTS BY UNITS SOLD


,product,total_revenue,units_sold
0,Headphones,4322500,665
2,Power Bank,1570000,628
4,Smartphone,7159500,387


### Task 4: Top Products by Revenue and Units Sold

**Top 3 products by revenue:**
1. Laptop — KSh 9,300,000.
2. Smartphone — KSh 7,159,500.
3. Smart TV — KSh 5,355,000.

**Top 3 products by units sold:**
1. Headphones — 665 units.
2. Power Bank — 628 units.
3. Smartphone — 387 units.

**Explanation:** Laptops generated the highest revenue because they have a high unit price, even though only 150 units were sold. Headphones and Power Banks sold more units but generated less revenue because their unit prices are lower. Smartphones performed strongly in both rankings, showing that they sold many units and generated substantial revenue.

In [22]:

# Compare weekend and weekday sales
weekend_summary = df.groupby("is_weekend").agg(
    total_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
).reset_index()

weekend_summary["day_type"] = weekend_summary["is_weekend"].map({
    False: "Weekday",
    True: "Weekend"
})

display(weekend_summary[
    ["day_type", "total_orders", "total_revenue", "average_order_revenue"]
])


,day_type,total_orders,total_revenue,average_order_revenue
0,Weekday,721,23623000,32764.216366
1,Weekend,279,9012000,32301.075269


### Task 5: Weekday vs Weekend Sales

- **Weekdays:** 721 orders generated KSh 23,623,000 in total revenue, with an average order revenue of KSh 32,764.22.
- **Weekends:** 279 orders generated KSh 9,012,000 in total revenue, with an average order revenue of KSh 32,301.08.

**Explanation:** Weekdays generated more total revenue because they had 442 more orders than weekends. However, the average revenue per order was slightly higher on weekdays, by approximately KSh 463.14. This suggests that the difference in total revenue was mainly driven by the larger number of weekday orders.

In [23]:

# Monthly sales analysis
monthly_summary = df.groupby("month").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count")
).reset_index()

# Arrange months in calendar order
month_order = [
    "January", "February", "March",
    "April", "May", "June"
]

monthly_summary["month"] = pd.Categorical(
    monthly_summary["month"],
    categories=month_order,
    ordered=True
)

monthly_summary = monthly_summary.sort_values("month")

display(monthly_summary)

# Identify best and worst months
best_month = monthly_summary.loc[
    monthly_summary["total_revenue"].idxmax()
]

worst_month = monthly_summary.loc[
    monthly_summary["total_revenue"].idxmin()
]

percentage_difference = (
    (best_month["total_revenue"] - worst_month["total_revenue"])
    / worst_month["total_revenue"]
) * 100

print("\nBest month:", best_month["month"])
print("Best month revenue:", best_month["total_revenue"])

print("\nWorst month:", worst_month["month"])
print("Worst month revenue:", worst_month["total_revenue"])

print(
    "Percentage difference relative to the worst month:",
    round(percentage_difference, 2), "%"
)


,month,total_revenue,total_orders
2,January,5444000,172
1,February,5212000,156
4,March,6004500,186
0,April,5526000,167
5,May,4993000,153
3,June,5455500,166



Best month: March
Best month revenue: 6004500

Worst month: May
Worst month revenue: 4993000
Percentage difference relative to the worst month: 20.26 %


### Task 6: Monthly Revenue Analysis

- **Best month:** March generated the highest revenue of KSh 6,004,500 from 186 orders.
- **Worst month:** May generated the lowest revenue of KSh 4,993,000 from 153 orders.
- **Percentage difference:** March revenue was 20.26% higher than May revenue, using May as the baseline.

**Explanation:** March was the strongest month in the first half of 2026, while May recorded the lowest revenue. The difference may reflect changes in the number of orders, products purchased, or average order values. Further investigation would be needed to determine the exact causes.

In [24]:

# Filter Premium online orders from Nairobi and Mombasa
premium_online = df[
    (df["price_band"] == "Premium") &
    (df["channel"] == "Online") &
    (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

# Save the filtered data without an index column
os.makedirs("/content/output", exist_ok=True)

output_path = "/content/output/premium_online_coast_capital.csv"
premium_online.to_csv(output_path, index=False)

# Report the number of orders and total revenue
print("Number of matching orders:", len(premium_online))
print("Total revenue: KSh", premium_online["revenue"].sum())
print("Saved to:", output_path)

display(premium_online.head())


Number of matching orders: 80
Total revenue: KSh 5079000
Saved to: /content/output/premium_online_coast_capital.csv


,order_id,order_date,branch,product,channel,units,unit_price,revenue,month,weekday,is_weekend,price_band
22,DK0463,2026-01-05,Nairobi,Laptop,Online,2,62000,124000,January,Monday,False,Premium
32,DK0336,2026-01-06,Nairobi,Laptop,Online,1,62000,62000,January,Tuesday,False,Premium
40,DK0808,2026-01-07,Mombasa,Smart TV,Online,2,45000,90000,January,Wednesday,False,Premium
45,DK0119,2026-01-08,Nairobi,Tablet,Online,1,28000,28000,January,Thursday,False,Premium
91,DK0904,2026-01-17,Nairobi,Laptop,Online,2,62000,124000,January,Saturday,True,Premium


In [25]:

import os

# Create the exact folder required by the assignment
os.makedirs("/content/outputs", exist_ok=True)

# Save the filtered data without the extra index column
output_path = "/content/outputs/premium_online_coast_capital.csv"
premium_online.to_csv(output_path, index=False)

print("Number of matching orders:", len(premium_online))
print("Total revenue: KSh", premium_online["revenue"].sum())
print("File saved successfully to:", output_path)


Number of matching orders: 80
Total revenue: KSh 5079000
File saved successfully to: /content/outputs/premium_online_coast_capital.csv


### Task 7: Premium Online Orders from Nairobi and Mombasa

- A total of **80 orders** matched the Premium price band, Online channel, and Nairobi or Mombasa branch conditions.
- The matching orders generated total revenue of **KSh 5,079,000**.
- The filtered data was saved as `outputs/premium_online_coast_capital.csv` without an extra index column.

In [26]:

import os

excel_path = "/content/excel/duka_branch_summary.xlsx"

if os.path.exists(excel_path):
    print("Excel file found!")
    print("File location:", excel_path)
else:
    print("Excel file not found at this location.")


Excel file found!
File location: /content/excel/duka_branch_summary.xlsx


In [27]:

import os

for root, folders, files in os.walk("/content"):
    for file in files:
        if file == "duka_branch_summary.xlsx":
            print("Excel file found here:")
            print(os.path.join(root, file))


Excel file found here:
/content/excel/duka_branch_summary.xlsx


In [28]:

import pandas as pd

excel_path = "/content/excel/duka_branch_summary.xlsx"

excel_file = pd.ExcelFile(excel_path)

print("Sheets available in the Excel file:")
print(excel_file.sheet_names)


Sheets available in the Excel file:
['Branch_Summary', 'Data', 'Regions', 'Pivot']


In [29]:
import pandas as pd

file_path = "/content/excel/duka_branch_summary.xlsx"

excel_file = pd.ExcelFile(file_path)

for sheet in excel_file.sheet_names:
    print(f"\n--- {sheet} ---")
    df = pd.read_excel(file_path, sheet_name=sheet)
    print(df.head())
    print("Shape:", df.shape)


--- Branch_Summary ---
    branch  total_revenue_kes  orders  revenue_rank       region  Unnamed: 5  \
0  Nairobi           12133000     349           1.0      Capital         NaN   
1  Mombasa            8444500     263           2.0        Coast         NaN   
2   Nakuru            4392000     128           3.0  Rift Valley         NaN   
3  Eldoret            3860000     115           4.0  Rift Valley         NaN   
4   Kisumu            3805500     145           5.0      Western         NaN   

                                               Notes  
0  revenue = units x unit_price (Data!H). Region ...  
1  PivotTable of revenue by branch and channel is...  
2                                                NaN  
3                                                NaN  
4                                                NaN  
Shape: (6, 7)

--- Data ---
  order_id  order_date   branch     product   channel  units  unit_price  \
0   DK0118  2026-01-01  Mombasa  Headphones  In-Store      4 

In [30]:
import pandas as pd

file_path = "/content/excel/duka_branch_summary.xlsx"

df = pd.read_excel(file_path, sheet_name="Data")

# Calculate revenue for each branch
branch_sales = df.groupby("branch")["revenue"].sum()

print("TOTAL REVENUE BY BRANCH")
print(branch_sales.sort_values(ascending=False))

print("\nTOTAL SALES REVENUE:")
print(f"KSh {df['revenue'].sum():,.2f}")

TOTAL REVENUE BY BRANCH
branch
Nairobi    12133000
Mombasa     8444500
Nakuru      4392000
Eldoret     3860000
Kisumu      3805500
Name: revenue, dtype: int64

TOTAL SALES REVENUE:
KSh 32,635,000.00


In [31]:
# Calculate revenue by sales channel
channel_sales = df.groupby("channel")["revenue"].sum()

print("TOTAL REVENUE BY CHANNEL")
print(channel_sales.sort_values(ascending=False))

print("\nNUMBER OF ORDERS BY CHANNEL")
print(df["channel"].value_counts())

TOTAL REVENUE BY CHANNEL
channel
In-Store    19217500
Online      13417500
Name: revenue, dtype: int64

NUMBER OF ORDERS BY CHANNEL
channel
In-Store    586
Online      414
Name: count, dtype: int64


In [32]:
# Calculate revenue and units sold by product
product_sales = df.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    total_units=("units", "sum")
)

print("TOP 5 PRODUCTS BY REVENUE")
print(product_sales.sort_values(
    by="total_revenue", ascending=False
).head(5))

print("\nTOTAL UNITS SOLD:")
print(df["units"].sum())

TOP 5 PRODUCTS BY REVENUE
            total_revenue  total_units
product                               
Laptop            9300000          150
Smartphone        7159500          387
Smart TV          5355000          119
Tablet            4928000          176
Headphones        4322500          665

TOTAL UNITS SOLD:
2125


In [33]:
# Convert order_date to datetime
df["order_date"] = pd.to_datetime(df["order_date"])

# Extract month from order date
df["month"] = df["order_date"].dt.month

# Calculate monthly revenue
monthly_sales = df.groupby("month")["revenue"].sum()

print("TOTAL REVENUE BY MONTH")
print(monthly_sales)

print("\nMONTH WITH HIGHEST REVENUE:")
print(monthly_sales.idxmax())

print("\nHIGHEST MONTHLY REVENUE:")
print(f"KSh {monthly_sales.max():,.2f}")

TOTAL REVENUE BY MONTH
month
1    5444000
2    5212000
3    6004500
4    5526000
5    4993000
6    5455500
Name: revenue, dtype: int64

MONTH WITH HIGHEST REVENUE:
3

HIGHEST MONTHLY REVENUE:
KSh 6,004,500.00


In [34]:
print("DUKA SALES BUSINESS SUMMARY")
print("=" * 35)

print(f"Total orders: {len(df)}")
print(f"Total revenue: KSh {df['revenue'].sum():,.2f}")
print(f"Total units sold: {df['units'].sum():,}")
print(f"Number of branches: {df['branch'].nunique()}")
print(f"Number of products: {df['product'].nunique()}")
print(f"Top branch: {branch_sales.idxmax()}")
print(f"Top product by revenue: {product_sales['total_revenue'].idxmax()}")
print(f"Top sales channel: {channel_sales.idxmax()}")
print(f"Best month: {monthly_sales.idxmax()}")

DUKA SALES BUSINESS SUMMARY
Total orders: 1000
Total revenue: KSh 32,635,000.00
Total units sold: 2,125
Number of branches: 5
Number of products: 6
Top branch: Nairobi
Top product by revenue: Laptop
Top sales channel: In-Store
Best month: 3


In [35]:
summary = pd.DataFrame({
    "metric": [
        "Total orders",
        "Total revenue",
        "Total units sold",
        "Number of branches",
        "Number of products",
        "Top branch",
        "Top product by revenue",
        "Top sales channel",
        "Best month"
    ],
    "value": [
        len(df),
        df["revenue"].sum(),
        df["units"].sum(),
        df["branch"].nunique(),
        df["product"].nunique(),
        branch_sales.idxmax(),
        product_sales["total_revenue"].idxmax(),
        channel_sales.idxmax(),
        monthly_sales.idxmax()
    ]
})

summary.to_csv("/content/duka_sales_summary.csv", index=False)

print(summary)
print("\nSummary saved successfully!")

                   metric     value
0            Total orders      1000
1           Total revenue  32635000
2        Total units sold      2125
3      Number of branches         5
4      Number of products         6
5              Top branch   Nairobi
6  Top product by revenue    Laptop
7       Top sales channel  In-Store
8              Best month         3

Summary saved successfully!
